<div align="center">

# Week 05: Feedback Control: P, PI and PID, Stability and Frequency Control

**Cyber-Physical Systems (BIL-435), hands-on Colab notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-05) · [Lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html) · [Interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lab.html) · [PDF notes](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/Week05_Lecture_Notes.pdf)

</div>

## How to use this Colab notebook

This Colab notebook is the hands-on part of the week. The concepts are explained on the [lecture page](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows. They have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from an orange Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures. When the Colab notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback. It also stores the constants of the island of the campus. They are the inertia and the damping, the lag and the limit of the inverter, the noise of the frequency meter and the two load steps. The second cell keeps the expected values of the exercises in the table `ANSWERS`, with its code hidden, so that no answer stands next to its exercise.

In [ ]:
# Environment: Everything used by this course is preinstalled in Google Colab.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import io
import math
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
plt.rcParams.update({"figure.dpi": 110, "figure.figsize": (7, 3.6),
                     "axes.grid": True, "grid.alpha": 0.3})

def check(name, value, expected=None, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback.
    Without an expected value in the call, check takes the value and its tolerance
    from ANSWERS, the table of the next cell."""
    if expected is None:
        answer, tol = ANSWERS[name]
        expected = answer()
    if value is None:
        print(f"{name}: not answered yet. Replace None and run the cell again.")
        return False
    try:
        scale = max(1.0, abs(float(expected)))
        ok = abs(float(value) - float(expected)) <= tol * scale
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else
                         "not yet. Compare your steps with the lecture."))
    return ok

# The island of the campus microgrid (Weeks 2 and 4)
M_KWS, D_KW = 80.0, 12.0                     # inertia in kW s/Hz and damping of the loads in kW/Hz
TAU_A, P_MAX = 0.2, 250.0                    # time constant of the inverter in s and its limit in kW
SIGMA_F, TS = 0.025, 0.01                    # noise of one frequency reading in Hz, a reading every 10 ms
STEP_CARS, STEP_WIND = 66.0, 240.0           # three cars start charging, the wind turbine trips
print("Setup complete. Running in Colab:", IN_COLAB, "| M", M_KWS, "kW s/Hz | D", D_KW, "kW/Hz | inverter",
      TAU_A, "s and", P_MAX, "kW")

In [ ]:
#@title Expected answers of the exercises (run this cell; open it only after trying the exercises)
ANSWERS = {
    "Exercise 5.1": (lambda: 60.0, 1e-9),
    "Exercise 5.2": (lambda: -250.0, 1e-9),
    "Exercise 5.3": (lambda: 20.0, 1e-9),
    "Exercise A": (lambda: -66 / (12 + 300), 1e-6),
    "Exercise B": (lambda: 137 / (2 * math.sqrt(80 * 120)), 1e-6),
    "Exercise C": (lambda: (80 + 12 * 0.1) * 137 / (80 * 0.1), 1e-6),
    "Exercise D": (lambda: 20 * math.sqrt(2) * 0.025 / 0.01, 1e-6),
    "Exercise E": (lambda: float(STAGE5['PID 125/60/40: within 20 mHz after (s)']), 1e-9),
}

## Python step 5: Functions for controllers

A controller is a function: It takes a measurement and returns a command. This step writes such functions with default arguments and with several results. It then gives a function a memory, as the integral of a controller needs, and runs a step response in a loop &#91;[2](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#ref-2)&#93;.

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STEP_WIND" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

### Default and keyword arguments

In [ ]:
# Python practice: a small example for learning the language
def p_control(error_hz, kp=125.0):
    """Proportional control: the power of the battery in kW for a frequency error in Hz."""
    return kp * error_hz

print(p_control(0.2))                # the default gain of 125 kW/Hz
print(p_control(0.2, kp=250.0))      # a keyword argument overrides the default

`kp=125.0` gives the argument a default value, which the first call uses. The second call names the argument and overrides it. A named argument makes a call readable, because `kp=250.0` says what the number means.

### Returning several values

In [ ]:
# Python practice: a small example for learning the language
def limit(u_kw, u_max=250.0):
    """Clips a command to the limits of the inverter and reports whether it was clipped."""
    clipped = max(-u_max, min(u_max, u_kw))
    return clipped, clipped != u_kw

u, saturated = limit(310.0)
print(u, saturated)

A function can return several values separated by commas. Python packs them into a tuple, and `u, saturated = ...` unpacks the tuple into two names.

### Functions that remember: Closures

In [ ]:
# Python practice: a small example for learning the language
def make_integrator(ki, ts):
    """Returns a function that adds ki * ts * error to its sum at each call."""
    total = 0.0
    def integrate(error):
        nonlocal total
        total += ki * ts * error
        return total
    return integrate

integ = make_integrator(ki=60.0, ts=0.01)
for e in [0.2, 0.2, 0.2]:
    print(round(integ(e), 3))

`make_integrator` builds the inner function `integrate` and returns it. The inner function keeps access to `total` after `make_integrator` has finished, which makes it a *closure*. The word `nonlocal` allows it to change `total`, so the sum survives from one call to the next, as the integral of a controller must.

### A step response in a loop

In [ ]:
# Python practice: a small example for learning the language
def step_response(controller, dP=66.0, T=10.0, dt=0.01, M=80.0, D=12.0):
    """Euler steps of the island with a controller that maps the error to the power of the battery."""
    f, trace = 0.0, []
    for _ in range(int(T / dt)):
        pb = controller(-f)                  # the error is the set point 0 minus the deviation
        f += dt * (pb - dP - D * f) / M
        trace.append(f)
    return np.array(trace)

df = step_response(p_control)
print("final deviation with P control:", round(float(df[-1]), 3), "Hz")

A function can be passed to another function like any other value. The function `step_response` calls whatever controller it receives, so the same loop tests P control here and any other controller later.

**Quick check.** What does `integ(0.0)` return after the three calls above, and why?

<details><summary>Answer</summary>

0.36: An error of zero adds nothing, but the sum remembers the earlier calls.

</details>

### Exercises for Python step 5

**Exercise 5.1.** Store the power that `p_control` returns for an error of 0.3 Hz with the gain 200 kW/Hz in `u_51`.

In [ ]:
u_51 = None   # your answer

check("Exercise 5.1", u_51)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
u_51 = p_control(0.3, kp=200.0)

check("Exercise 5.1", u_51)

**Exercise 5.2.** Use `limit` to clip a command of -400 kW and store the clipped value in `u_52`.

In [ ]:
u_52 = None   # your answer

check("Exercise 5.2", u_52)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
u_52, _ = limit(-400.0)

check("Exercise 5.2", u_52)

**Exercise 5.3.** Build an integrator with `ki=10.0` and `ts=0.5`, call it with the errors 1.0 and 3.0 and store the second result in `i_53`.

In [ ]:
i_53 = None   # your answer

check("Exercise 5.3", i_53)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
I53 = make_integrator(10.0, 0.5)
I53(1.0)
i_53 = I53(3.0)

check("Exercise 5.3", i_53)

## 1. Proportional control: The droop and its error &#91;[3](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#ref-3)&#93;

*Lecture: [Feedback: Proportional control and its error](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#feedback)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "step_response" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The droop of Week 2 is a proportional controller. Python simulates the island with the lag of the inverter for three proportional gains and shows the deviation that each of them leaves.

**From the formulas to the code.** The swing equation $M\,\dot{\Delta f} = P_b - \Delta P_{load} - D\,\Delta f$ is `dfdt`, the inverter $\tau_a \dot{P}_b = u - P_b$ is `dpb`, and the command $u = -K_p \Delta f + I - K_d \dot{\Delta f}$ is `u`. The function `simulate` takes steps of 2 ms with the Runge-Kutta method. The remaining deviation $-\Delta P_{load}/(D + K_p)$ is printed next to the simulation.

In [ ]:
def simulate(kp, ki=0.0, kd=0.0, step_kw=STEP_CARS, anti_windup=True, T=30.0, dt=0.002):
    """The island after a load step: deviation in Hz, power of the battery and command in kW, every dt seconds."""
    def rhs(f, pb, i_kw):
        dfdt = (pb - step_kw - D_KW * f) / M_KWS                # the swing equation of Week 2
        u = -kp * f + i_kw - kd * dfdt                          # the command: P, I and D terms
        di = -ki * f                                            # the integral of the error e = -f
        if anti_windup and abs(u) > P_MAX and di * u > 0:
            di = 0.0                                            # the integral stops at the limit
        dpb = (min(P_MAX, max(-P_MAX, u)) - pb) / TAU_A         # the inverter follows its limited command
        return dfdt, dpb, di, u
    n = int(round(T / dt))
    f = pb = i_kw = 0.0
    F, PB, U = np.zeros(n + 1), np.zeros(n + 1), np.zeros(n + 1)
    U[0] = rhs(f, pb, i_kw)[3]
    for k in range(n):                                          # Runge-Kutta steps of dt
        a = rhs(f, pb, i_kw)
        b = rhs(f + dt / 2 * a[0], pb + dt / 2 * a[1], i_kw + dt / 2 * a[2])
        c = rhs(f + dt / 2 * b[0], pb + dt / 2 * b[1], i_kw + dt / 2 * b[2])
        d = rhs(f + dt * c[0], pb + dt * c[1], i_kw + dt * c[2])
        f += dt / 6 * (a[0] + 2 * b[0] + 2 * c[0] + d[0])
        pb += dt / 6 * (a[1] + 2 * b[1] + 2 * c[1] + d[1])
        i_kw += dt / 6 * (a[2] + 2 * b[2] + 2 * c[2] + d[2])
        F[k + 1], PB[k + 1], U[k + 1] = f, pb, rhs(f, pb, i_kw)[3]
    return np.arange(n + 1) * dt, F, PB, U

def metrics(t, F, band=0.02):
    """Nadir, overshoot above 50 Hz, settling time within the band and final deviation of a run."""
    dips = np.where((F[1:-1] < F[:-2]) & (F[1:-1] <= F[2:]))[0] + 1
    nadir = F[dips[0]] if len(dips) else F.min()
    outside = np.where(np.abs(F) > band)[0]
    if len(outside) and t[outside[-1]] > t[-1] - 2.0:
        settle = None                                           # still outside the band in the last 2 s
    else:
        settle = t[outside[-1] + 1] if len(outside) else 0.0
    return dict(nadir=round(float(nadir), 3), overshoot=round(max(0.0, float(F.max())), 3),
                settle=None if settle is None else round(float(settle), 2), final=round(float(F[-1]), 3))

for kp in [50.0, 125.0, 250.0]:
    t, F, PB, U = simulate(kp)
    print(f"Kp = {kp:5.0f} kW/Hz: {metrics(t, F)} | formula {-STEP_CARS / (D_KW + kp):.3f} Hz")

**Python note.** `rhs` returns the rates of change of the three states, the deviation, the power of the battery and the integral, together with the command. The Runge-Kutta method averages four such slopes for each step and is far more accurate than the Euler step of Week 2 for the same step size. `metrics` finds the first dip of the deviation, its overshoot above 50 Hz and the last time it leaves the band of 20 mHz. Its `settle` is `None` when the deviation is still outside the band in the last 2 s of the run.

**Exercise A.** Store the deviation in Hz at which P control with `kp=300` settles after the step of 66 kW, from the formula, in `df_a`.

In [ ]:
df_a = None   # your answer

check("Exercise A", df_a)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
df_a = -STEP_CARS / (D_KW + 300.0)

check("Exercise A", df_a)

In [ ]:
# Figure: the frequency deviation after 66 kW for three proportional gains
fig, ax = plt.subplots(figsize=(8, 3.4))
for kp, col in [(50.0, "#B53F2E"), (125.0, "#1E6FA8"), (250.0, "#2E8B57")]:
    t, F, PB, U = simulate(kp, T=10.0)
    ax.plot(t, F, color=col, label=f"Kp = {kp:.0f} kW/Hz")
    ax.axhline(-STEP_CARS / (D_KW + kp), color=col, ls=":", lw=1)
ax.set_xlabel("time after the step (s)")
ax.set_ylabel("frequency deviation (Hz)")
ax.legend(fontsize=8)
plt.show()

**Observe and reflect.** Why does a larger gain leave a smaller deviation, but never none? What would the battery need to remove it?

## 2. Integral action: Your runs on the bench &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#ref-1)&#93;

*Lecture: [Integral action: Back to 50 Hz](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#pi-control)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "metrics" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** An integral term keeps changing the power as long as any deviation remains, and together with the proportional term it forms a proportional-integral (PI) controller. Python computes the natural angular frequency and the damping ratio of four integral gains and simulates each with the lag of the inverter. It then checks your runs on the hardware-in-the-loop bench.

**From the formulas to the code.** The natural angular frequency $\omega_n = \sqrt{K_i/M}$ and the damping ratio $\zeta = (D + K_p)/(2\sqrt{M K_i})$ are `wn` and `zeta`. The integral term $K_i \int e\,dt$ is the third state `i_kw` of `simulate`, whose rate of change is $K_i\,e = -K_i\,\Delta f$.

In [ ]:
for ki in [30.0, 60.0, 120.0, 240.0]:
    wn = math.sqrt(ki / M_KWS)
    zeta = (D_KW + 125.0) / (2 * math.sqrt(M_KWS * ki))
    t, F, PB, U = simulate(125.0, ki)
    print(f"Ki = {ki:5.0f}: wn {wn:.3f} rad/s, zeta {zeta:.3f} | {metrics(t, F)}")

**Python note.** Each line prints the dictionary of `metrics` with its keys: The nadir, the overshoot above 50 Hz, the settling time within 20 mHz and the final deviation. A damping ratio near 1 returns without overshoot, while smaller values swing past 50 Hz.

**Exercise B.** Store the damping ratio of the PI controller with `kp=125` and `ki=120`, without the lag of the inverter, in `zeta_b`.

In [ ]:
zeta_b = None   # your answer

check("Exercise B", zeta_b)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
zeta_b = (D_KW + 125.0) / (2 * math.sqrt(M_KWS * 120.0))

check("Exercise B", zeta_b)

Part A of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lab.html) tests the controller on the hardware-in-the-loop bench. Paste the copied rows into `MY_RUNS`, and the code runs the model with your gains and compares it with each run.

In [ ]:
# Paste the results of lab Part A between the triple quotes.
MY_RUNS = """
"""
if MY_RUNS.strip():
    runs = pd.read_csv(io.StringIO(MY_RUNS.strip()))
    source = "your own runs on the bench"
else:
    runs = pd.DataFrame({"run": [1, 2, 3, 4, 5], "kp": [125, 250, 125, 125, 125], "ki": [0, 0, 60, 700, 710],
                         "kd": [0, 0, 0, 0, 0], "step_kw": [66, 66, 66, 66, 66],
                         "nadir_hz": [-0.484, -0.284, -0.413, -0.276, -0.275], "settle_s": [None, None, 6.05, None, None]})
    source = "a simulated student"
model = [metrics(*simulate(r.kp, r.ki, r.kd, r.step_kw)[:2]) for r in runs.itertuples()]
runs["model_nadir"] = [m["nadir"] for m in model]
runs["model_settle"] = [m["settle"] for m in model]
print("Data:", source)
print(runs[["run", "kp", "ki", "kd", "step_kw", "nadir_hz", "model_nadir", "settle_s", "model_settle"]].to_string(index=False))

In [ ]:
# Figure: the frequency deviation and the power of the battery for four integral gains
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for ki, col in [(0.0, "#56616D"), (30.0, "#B53F2E"), (60.0, "#1E6FA8"), (240.0, "#2E8B57")]:
    t, F, PB, U = simulate(125.0, ki, T=15.0)
    ax[0].plot(t, F, color=col, label=f"Ki = {ki:.0f}")
    ax[1].plot(t, PB, color=col)
ax[0].set_xlabel("time after the step (s)")
ax[0].set_ylabel("frequency deviation (Hz)")
ax[1].set_xlabel("time after the step (s)")
ax[1].set_ylabel("power of the battery (kW)")
ax[0].legend(fontsize=8)
plt.show()

**Observe and reflect.** Which of your runs on the bench returned fastest without overshoot? Why does the battery deliver exactly 66 kW at the end, whatever the integral gain?

## 3. Stability: Poles and the limit of the integral &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#ref-1)&#93;

*Lecture: [Stability: How far the integral may go](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#stability)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "zeta" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** With the lag of the inverter, the loop becomes a system of third order, and a large integral gain makes it oscillate. Python finds the poles of the loop and compares the limit of the Routh-Hurwitz criterion with simulations just below and just above it.

**From the formulas to the code.** The coefficients $a_3 = M\tau_a$, $a_2 = M + D\tau_a + K_d$, $a_1 = D + K_p$ and $a_0 = K_i$ form the list `coeffs`, and `np.roots(coeffs)` returns the poles. The limit $K_i^* = a_2 a_1/a_3$ is `ki_limit`, and $\omega^* = \sqrt{a_1/a_3}$ is `w_star`.

In [ ]:
def poles(kp, ki, kd=0.0, tau_a=TAU_A):
    """The roots of the characteristic equation of the loop with the lag of the inverter."""
    coeffs = [M_KWS * tau_a, M_KWS + D_KW * tau_a + kd, D_KW + kp, ki]
    return np.roots(coeffs)

def ki_limit(kp, kd=0.0, tau_a=TAU_A):
    """The largest stable integral gain by the Routh-Hurwitz criterion, a2 * a1 / a3."""
    return (M_KWS + D_KW * tau_a + kd) * (D_KW + kp) / (M_KWS * tau_a)

w_star = math.sqrt((D_KW + 125.0) / (M_KWS * TAU_A))
print(f"limit Ki* = {ki_limit(125.0):.2f} kW/(Hz s), oscillation {w_star:.3f} rad/s, period {2 * math.pi / w_star:.3f} s")
for ki in [60.0, 600.0, 705.0, 706.0, 800.0]:
    p = poles(125.0, ki)
    t, F, PB, U = simulate(125.0, ki)
    print(f"Ki = {ki:5.0f}: largest real part {p.real.max():+.4f} 1/s, "
          f"{'stable' if p.real.max() < 0 else 'unstable'} | swing in the last 5 s {np.abs(F[-2500:]).max():.3f} Hz")

**Python note.** `np.roots` takes the coefficients from the highest power down and returns the complex roots. `p.real.max()` is the real part of the slowest pole, which decides whether the loop is stable. `F[-2500:]` holds the last 5 s of the run, 2500 steps of 2 ms.

**Exercise C.** Store the limit of the integral gain for `kp=125`, `kd=0` and a faster inverter with `tau_a=0.1` in `ki_c`.

In [ ]:
ki_c = None   # your answer

check("Exercise C", ki_c)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
ki_c = ki_limit(125.0, 0.0, 0.1)

check("Exercise C", ki_c)

In [ ]:
# Figure: the deviation below, near and above the limit, and the poles as Ki grows
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for ki, col in [(600.0, "#1E6FA8"), (705.0, "#C98510"), (800.0, "#B53F2E")]:
    t, F, PB, U = simulate(125.0, ki, T=20.0)
    ax[0].plot(t, F, color=col, lw=1.2, label=f"Ki = {ki:.0f}")
ax[0].set_xlabel("time after the step (s)")
ax[0].set_ylabel("frequency deviation (Hz)")
ax[0].legend(fontsize=8)
kis = np.linspace(1, 1000, 200)
P = np.array([poles(125.0, ki) for ki in kis])
for j in range(3):
    ax[1].scatter(P[:, j].real, P[:, j].imag, c=kis, cmap="viridis", s=6)
ax[1].axvline(0, color="#56616D", lw=1)
ax[1].set_xlabel("real part (1/s)")
ax[1].set_ylabel("imaginary part (rad/s)")
plt.show()

**Observe and reflect.** How close to the limit did your runs on the bench oscillate steadily? Which part of the loop would you make faster to raise the limit?

## 4. Derivative, noise and windup: Your runs in the cabinet &#91;[1](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#ref-1)&#93;

*Lecture: [Saturation and windup](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#windup)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "ki_limit" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** The derivative term reacts to the rate of change, and the inverter saturates at 250 kW. Python measures the noise that the derivative adds with raw and with filtered readings, and it runs the trip of the wind turbine with and without anti-windup. Finally, it writes the controller as a function that is called 100 times a second.

**From the formulas to the code.** The noise of the raw derivative $\sqrt{2}\sigma/T_s$ and of the filtered one $\sqrt{2}(\sigma/\sqrt{7})/0.1$ are compared with simulated readings of noise `SIGMA_F`. The anti-windup is the condition `abs(u) > P_MAX and di * u > 0` inside `simulate`: It stops the integral while the command lies beyond the limit and the error would drive it further.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
readings = rng.normal(0.0, SIGMA_F, 20000)                    # 200 s of readings of a steady 50 Hz, every 10 ms
raw = np.diff(readings) / TS                                  # the derivative from two consecutive readings
avg = np.convolve(readings, np.ones(7) / 7, mode="valid")     # the moving average of Week 4
filt = (avg[10:] - avg[:-10]) / (10 * TS)                     # the derivative over 0.1 s
for name, d, formula in [("raw", raw, math.sqrt(2) * SIGMA_F / TS),
                         ("filtered", filt, math.sqrt(2) * SIGMA_F / math.sqrt(7) / 0.1)]:
    print(f"{name:>8}: noise {d.std():.4f} Hz/s (formula {formula:.4f}), with Kd = 40: {40 * d.std():6.2f} kW")
for kd in [0.0, 20.0, 40.0, 80.0]:
    t, F, PB, U = simulate(125.0, 60.0, kd)
    print(f"Kd = {kd:4.0f} kW s/Hz: {metrics(t, F)}")

**Python note.** `np.diff` returns the differences of neighbouring values. Two readings only 10 ms apart differ mostly by their noise, which the division by 0.01 s multiplies by 100. The moving average and the longer distance of 0.1 s reduce the noise by a factor of about 26.

**Exercise D.** Store the noise in kW that a derivative gain of 20 kW s/Hz puts on the command with raw readings, from the formula, in `sigma_d`.

In [ ]:
sigma_d = None   # your answer

check("Exercise D", sigma_d)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
sigma_d = 20.0 * math.sqrt(2) * SIGMA_F / TS

check("Exercise D", sigma_d)

In [ ]:
for aw in [False, True]:
    t, F, PB, U = simulate(125.0, 60.0, step_kw=STEP_WIND, anti_windup=aw)
    beyond = np.sum(np.abs(U) > P_MAX + 1e-6) * (t[1] - t[0])
    print(f"anti-windup {'on ' if aw else 'off'}: {metrics(t, F)} | largest command {U.max():.1f} kW, "
          f"beyond the limit for {beyond:.2f} s")

def make_pid(kp, ki, kd, ts=TS, u_max=P_MAX):
    """A discrete PID controller with a limit and an anti-windup, called once every ts seconds."""
    integral, last = 0.0, None
    def pid(error):
        nonlocal integral, last
        derivative = 0.0 if last is None else (error - last) / ts
        last = error
        u = kp * error + integral + kd * derivative
        if not (abs(u) > u_max and error * u > 0):
            integral += ki * ts * error
        return min(u_max, max(-u_max, u))
    return pid

pid = make_pid(125.0, 60.0, 0.0)
f, pb, dt, trace = 0.0, 0.0, 0.002, []
for k in range(int(30 / dt)):
    if k % 5 == 0:                                            # the controller runs every 10 ms
        u = pid(-f)
    pb += dt * (u - pb) / TAU_A
    f += dt * (pb - STEP_WIND - D_KW * f) / M_KWS
    trace.append(f)
trace = np.array(trace)
print("discrete PI at 100 Hz, trip of the wind turbine:", metrics(np.arange(1, len(trace) + 1) * dt, trace))

**Python note.** `make_pid` is a closure, as in Python step 5: `integral` and `last` survive from one call to the next. The loop calls the controller only every fifth step of 2 ms, and the command stays constant in between, as on a real processor. The discrete controller gives almost the same result as the continuous model.

Part C of the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lab.html) opens the inverter cabinet of the battery. Paste the copied rows into `MY_TRIPS`, and the code compares your runs with the model and with the formulas of the noise.

In [ ]:
# Paste the results of lab Part C between the triple quotes.
MY_TRIPS = """
"""
if MY_TRIPS.strip():
    trips = pd.read_csv(io.StringIO(MY_TRIPS.strip()))
    source = "your own runs in the inverter cabinet"
else:
    trips = pd.DataFrame({"run": [1, 2, 3, 4], "event_kw": [240, 240, 66, 66], "anti_windup": ["off", "on", "on", "on"],
                          "derivative": ["off", "off", "raw", "filtered"], "overshoot_hz": [0.372, 0.0, 0.032, 0.006],
                          "settle_s": [17.2, 10.16, None, 5.86], "d_noise_kw": [None, None, 148.97, 5.1]})
    source = "a simulated student"
def model_row(r):
    if r.derivative == "off":
        m = metrics(*simulate(125.0, 60.0, 0.0, r.event_kw, r.anti_windup == "on")[:2])
        return m["overshoot"], m["settle"], None
    sd = math.sqrt(2) * SIGMA_F / TS if r.derivative == "raw" else math.sqrt(2) * SIGMA_F / math.sqrt(7) / 0.1
    return None, None, round(40 * sd, 2)
model = [model_row(r) for r in trips.itertuples()]
trips["model_overshoot"] = [m[0] for m in model]
trips["model_settle"] = [m[1] for m in model]
trips["formula_noise_kw"] = [m[2] for m in model]
print("Data:", source)
print(trips.to_string(index=False))

In [ ]:
# Figure: the trip of the wind turbine with and without anti-windup
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for aw, col in [(False, "#B53F2E"), (True, "#1E6FA8")]:
    t, F, PB, U = simulate(125.0, 60.0, step_kw=STEP_WIND, anti_windup=aw)
    name = "with anti-windup" if aw else "without anti-windup"
    ax[0].plot(t, F, color=col, label=name)
    ax[1].plot(t, U, color=col, ls=":", lw=1)
    ax[1].plot(t, PB, color=col, label=name)
ax[1].axhline(P_MAX, color="#56616D", lw=1)
ax[0].set_xlabel("time after the trip (s)")
ax[0].set_ylabel("frequency deviation (Hz)")
ax[1].set_xlabel("time after the trip (s)")
ax[1].set_ylabel("command (dotted) and power (kW)")
ax[0].legend(fontsize=8)
plt.show()

**Observe and reflect.** How much did the anti-windup shorten the return to 50 Hz in your runs? Why would a raw derivative wear out the inverter?

## 5. The campus microgrid, stage 5

*Lecture: [The campus microgrid, stage 5: Frequency control by the battery](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#the-campus-microgrid-stage-5)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "make_pid" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Stage 5 designs the frequency controller of the battery for island mode, a proportional-integral-derivative (PID) controller. It keeps the droop of Week 2 as the proportional term and adds an integral and a filtered derivative. The command is limited to 250 kW, and an anti-windup protects the integral.

In [ ]:
STAGE5 = {}
STAGE5["droop alone: deviation after 66 kW (Hz)"] = metrics(*simulate(125.0)[:2])["final"]
m = metrics(*simulate(125.0, 60.0)[:2])
STAGE5["PI 125/60: nadir (Hz)"] = m["nadir"]
STAGE5["PI 125/60: within 20 mHz after (s)"] = m["settle"]
STAGE5["limit of Ki with Kd = 40 (kW/(Hz s))"] = round(ki_limit(125.0, 40.0), 2)
m = metrics(*simulate(125.0, 60.0, 40.0)[:2])
STAGE5["PID 125/60/40: nadir after 66 kW (Hz)"] = m["nadir"]
STAGE5["PID 125/60/40: within 20 mHz after (s)"] = m["settle"]
STAGE5["noise of the derivative term, raw (kW)"] = round(40 * math.sqrt(2) * SIGMA_F / TS, 2)
STAGE5["noise of the derivative term, filtered (kW)"] = round(40 * math.sqrt(2) * SIGMA_F / math.sqrt(7) / 0.1, 2)
for aw in [False, True]:
    m = metrics(*simulate(125.0, 60.0, 40.0, STEP_WIND, aw)[:2])
    key = "with" if aw else "without"
    STAGE5[f"trip of 240 kW {key} anti-windup: overshoot (Hz)"] = m["overshoot"]
    STAGE5[f"trip of 240 kW {key} anti-windup: within 20 mHz after (s)"] = m["settle"]
for key, value in STAGE5.items():
    print(f"{key:<60} {value}")

**Python note.** `metrics(*simulate(...)[:2])` passes the first two results of `simulate`, the times and the deviation, to `metrics`. The star unpacks the tuple into two arguments.

In [ ]:
# Figure: the final controller after the load step and after the trip of the wind turbine
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for (kp, ki, kd), col, name in [((125.0, 0.0, 0.0), "#56616D", "droop alone"), ((125.0, 60.0, 0.0), "#C98510", "PI"),
                                ((125.0, 60.0, 40.0), "#1E6FA8", "final PID")]:
    t, F, PB, U = simulate(kp, ki, kd, T=15.0)
    ax[0].plot(t, F, color=col, label=name)
for aw, col in [(False, "#B53F2E"), (True, "#1E6FA8")]:
    t, F, PB, U = simulate(125.0, 60.0, 40.0, STEP_WIND, aw, T=20.0)
    ax[1].plot(t, F, color=col, label="with anti-windup" if aw else "without anti-windup")
for a in ax:
    a.set_xlabel("time (s)")
    a.set_ylabel("frequency deviation (Hz)")
    a.legend(fontsize=8)
plt.show()

**Exercise E.** Store the settling time of the final PID controller after the step of 66 kW, as kept in `STAGE5`, in `t_e`.

In [ ]:
t_e = None   # your answer

check("Exercise E", t_e)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
t_e = STAGE5["PID 125/60/40: within 20 mHz after (s)"]

check("Exercise E", t_e)

**Observe and reflect.** The final controller keeps a factor of 17.5 below the stability limit. Which result of stage 5 would you show the facility manager first, and why?

## 6. Your application: The application switch

*Lecture: [Your application](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lecture.html#your-application)*

**Started here?** If you opened the Colab notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures. This takes from a few seconds to a minute, and the cell does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "ANSWERS" in globals() and "STAGE5" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_url = ("https://raw.githubusercontent.com/utkukose/"
               "cyber-physical-systems-NB-lecture/main/weeks/week-05/"
               "NB05_feedback_control.ipynb")
    _cu_cells = json.load(urllib.request.urlopen(_cu_url))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            _cu_meta = _cu_c.get("metadata", {})
            if _cu_c["cell_type"] == "code" and _cu_meta.get("catchup", True):
                try:
                    _cu_src = _cu_ip.transform_cell("".join(_cu_c["source"]))
                    exec(_cu_src, _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    if _cu_err is None:
        print("The earlier sections have run. Continue with the next cell.")
    else:
        print(f"The catch-up stopped with an error ({_cu_err}). "
              "Run the notebook from the top instead.")

**Why Python here.** Feedback control runs wherever a quantity must follow a set point. The functions of this week tune and test a PI controller for a loop of first order after one change of a name.

The loops and their numbers are illustrative. Set `APP` to one of the six names and run the cell.

In [ ]:
APP = "cruise_control"
# choose: room_heating, cruise_control, motor_speed, water_tank, soil_moisture, battery_charger
APPS = {  # (quantity, unit, gain K, time constant in s, lag of the actuator in s, limit, unit of the actuator, step)
    "room_heating": ("temperature of a room", "K", 0.4, 43200.0, 600.0, 60.0, "kW", 2.0),
    "cruise_control": ("speed of a car", "m/s", 0.02, 30.0, 0.5, 4000.0, "N", 5.0),
    "motor_speed": ("speed of a motor", "rad/s", 30.0, 0.5, 0.01, 24.0, "V", 300.0),
    "water_tank": ("level of a water tank", "m", 0.02, 600.0, 5.0, 100.0, "l/min", 1.0),
    "soil_moisture": ("moisture of a soil", "%", 0.5, 3600.0, 60.0, 40.0, "l/min", 8.0),
    "battery_charger": ("current of a battery charger", "A", 2.0, 0.002, 0.0001, 100.0, "% duty", 32.0)}
desc, unit, K, tau, tau_a, u_max, u_unit, r = APPS[APP]
kp, ki = 4 / K, 4 / (K * tau)                  # the rule: the loop four times faster than the plant

def first_order(kp, ki, T=8 * tau, n=8000):
    dt, y, ua, i_sum, ys, us = T / n, 0.0, 0.0, 0.0, [], []
    for _ in range(n):
        e = r - y
        u = kp * e + i_sum
        if not (abs(u) > u_max and e * u > 0):
            i_sum += ki * e * dt
        u = max(-u_max, min(u_max, u))
        ua += dt * (u - ua) / tau_a            # the actuator follows its command
        y += dt * (K * ua - y) / tau
        ys.append(y)
        us.append(u)
    return np.array(ys), np.array(us), dt

def duration(x):
    return f"{x / 3600:.3g} h" if x >= 3600 else f"{x / 60:.3g} min" if x >= 60 else f"{x:.3g} s" if x >= 0.1 else f"{1000 * x:.3g} ms"

y_p, _, _ = first_order(kp, 0.0)
y_pi, u_pi, dt = first_order(kp, ki)
outside = np.where(np.abs(y_pi - r) > 0.02 * r)[0]
print(f"{APP}: {desc}, a step of {r:g} {unit}")
print(f"  P only (Kp = {kp:.3g} {u_unit} per {unit}): remaining error {r - y_p[-1]:.3g} {unit}")
print(f"  PI: within 2 percent after {duration((outside[-1] + 1) * dt)}, largest command {u_pi.max():.4g} {u_unit}"
      + (" at the limit" if u_pi.max() >= u_max else ""))

**Python note.** Each entry holds the quantity and its unit, the gain and the time constant of the plant, the lag and the limit of the actuator and the step. The rule $K_p = 4/K$ and $K_i = 4/(K\tau)$ makes the loop about four times faster than the plant. P control alone leaves a fifth of the step as an error, because $1/(1 + K K_p) = 1/5$.

**Observe and reflect.** Which loop reaches the limit of its actuator, and what would an anti-windup change there? Which loop would you rather control with P alone, and why?

## Application challenges

Each challenge transfers the ideas of the week to a field of computing. Choose the one closest to your interests and solve it in a copy of this Colab notebook.

| Field | Challenge |
|---|---|
| Software engineering | Write the controller as a class `PID` with the methods `update` and `reset`, and test it with errors whose correct outputs you compute by hand. |
| Embedded systems | Write the PI controller in integers only, with the frequency in millihertz and the power in watts, as a small microcontroller without floating point would run it. |
| Internet of Things | A thermostat receives its temperature over a wireless link that loses one reading in ten. What should its integral do with a missing reading? |
| Computer networks | Add a delay of the measurement to `simulate` and find the delay at which the PI controller of stage 5 begins to oscillate. Compare it with the delay margin of Week 8. |
| Control engineering | Add a delay of 0.3 s to the loop, find the ultimate gain and period of a P controller and apply the rules of Ziegler and Nichols. Compare the result with stage 5. |
| Data science | Simulate a step response with noise, then estimate M and D from it by least squares, as an engineer would from a recorded event. |
| Cybersecurity | An attacker raises the integral gain of the battery to 900. Write a monitor that detects the growing oscillation within two periods and switches back to the droop. |
| Artificial intelligence | Search the gains Kp, Ki and Kd that minimise the nadir plus the settling time on the simulator, with a random search of 200 trials that rejects every unstable set. |

## After the Colab notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/cyber-physical-systems-NB-lecture/weeks/week-05/lab.html), tabs *Check yourself* and *Reflect and export*. The weekly task and the research assignment are on the [week overview](https://github.com/utkukose/cyber-physical-systems-NB-lecture/tree/main/weeks/week-05).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

This course is updated in line with current developments in the field. First version: November 2026.

</div>